# Smart-Kage NOR postprocessing

This notebook creates the processed NOR analysis tables from three inputs:

1. `NOR_data.csv` files  
   These contain the timestamps when the wheels/drums changed.

2. `drum_output` files  
   These contain exploration events. Each row is one detected exploration sample.
   `timestamp` is seconds from midnight.
   `side` is left or right object exploration.

3. `kage_descriptions.csv`  
   This contains genotype, inclusion/exclusion, and which side was novel during the test phase.

The aim is to create one clean binned table that can be used for:
- cumulative exploration
- average exploration over time
- discrimination index over time
- first approach after wheel change during testing phase 

## Combine wheel-change timestamp files

Each `/kageX/NOR_data.csv` file contains the times when the wheels changed for that mouse.

This step finds all `NOR_data.csv` files, adds the Kage ID, and combines them into one file which tells us when each phase starts.

In [16]:
import numpy as np
import re
from pathlib import Path
from typing import List, Tuple, Optional
import pandas as pd
import sys

In [18]:
# Flag available for future conditional logging; currently not used.
DEBUG = True

# Active macOS path: update this to your data root containing kage* folders.
# ROOT_DIR = Path(r"/Volumes/Aimi_SmrtKg/2025-04-Apr-May_NLGF24m_Coh1/DB")
ROOT_DIR = Path(r"I:\SmartKages_Nov2025_onwards\2026-01-Jan-Feb_RELN-COLBOS_Coh2\DB")

# The output CSV name includes the parent folder name of ROOT_DIR for context.
# Example: if ROOT_DIR is /path/to/DB, parent_name might be 'to', producing 'to_NOR_timestamps_AllKages_Combined.csv'.
parent_name = ROOT_DIR.parent.name
OUTPUT_CSV = ROOT_DIR / f"{parent_name}_NOR_timestamps_AllKages_Combined.csv"
# --------------------------------

def extract_kage_id(path: Path) -> str:
    """
    Derive a normalized KageID (e.g., 'Kage1') from a folder path.
    - Looks for a pattern 'kage<digits>' (case-insensitive) in the folder name first.
    - If not found, searches upward through path parts.
    - Returns 'Kage_Unknown' if no match is found.

    Example:
    - 'kage1' -> 'Kage1'
    - 'KAGE 2' -> 'Kage2'
    """
    # Search the given folder's name for 'kage<number>'.
    m = re.search(r'kage\s*(\d+)', path.name, flags=re.IGNORECASE)
    if m:
        return f"Kage{m.group(1)}"
    # If not found in the immediate name, walk up the path parts in reverse to find a match.
    for part in path.parts[::-1]:
        m = re.search(r'kage\s*(\d+)', part, flags=re.IGNORECASE)
        if m:
            return f"Kage{m.group(1)}"
    # Fallback when the expected naming convention isn't present.
    return "Kage_Unknown"

def read_csv_with_fallbacks(csv_path: Path) -> Optional[pd.DataFrame]:
    """
    Robust CSV reader that:
    - Tries multiple common encodings: utf-8-sig, utf-8, cp1252, latin-1.
    - Uses 'sep=None' with the 'python' engine to infer delimiters (comma/semicolon/tab).
    - Skips malformed lines ('on_bad_lines="skip"') to avoid hard failures.

    Returns:
    - DataFrame if successfully read.
    - None if all attempts fail.
    """
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(
                csv_path,
                encoding=enc,
                sep=None,              # Infer delimiter automatically.
                engine="python",       # Required for delimiter inference.
                on_bad_lines="skip",   # Prevent read errors due to bad rows.
            )
        except Exception:
            # Try the next encoding if this attempt fails.
            continue
    # All attempts failed; log a warning and return None.
    print(f"Warning: Failed to read {csv_path}")
    return None

def find_kage_nor_csvs(root: Path) -> List[Tuple[str, Path]]:
    r"""
    Discover all NOR_data.csv files directly under kage* directories within 'root'.

    Returns:
    - List of tuples: (KageID, csv_path)
      Where KageID is derived via 'extract_kage_id', and csv_path is root/kageX/NOR_data.csv.

    Scans structure like:
      <root>/kage1/NOR_data.csv
      <root>/kage2/NOR_data.csv
      ...
    """
    results: List[Tuple[str, Path]] = []
    # Matches folder names like 'kage1', 'KAGE 2', etc.
    kage_dir_regex = re.compile(r'kage\s*\d+', flags=re.IGNORECASE)

    # Validate the root before scanning.
    if not root.exists():
        print(f"Root directory does not exist: {root}")
        return results

    # Iterate through subdirectories whose names match the 'kage<number>' pattern.
    for kage_dir in sorted(p for p in root.iterdir() if p.is_dir() and kage_dir_regex.match(p.name)):
        kage_id = extract_kage_id(kage_dir)
        csv_path = kage_dir / "NOR_data.csv"
        # Record only when NOR_data.csv exists; otherwise notify.
        if csv_path.exists():
            results.append((kage_id, csv_path))
        else:
            print(f"Note: No NOR_data.csv in {kage_dir}")
    return results

def main():
    """
    Orchestrates the consolidation:
    - Finds all candidate NOR_data.csv files.
    - Loads each with encoding/delimiter inference.
    - Injects 'KageID' and 'SourceFile' at the front of each DataFrame.
    - Concatenates and writes the combined CSV to OUTPUT_CSV.
    """
    # Locate all kage*/NOR_data.csv pairs under ROOT_DIR.
    pairs = find_kage_nor_csvs(ROOT_DIR)
    if not pairs:
        print(f"No CSV files found under: {ROOT_DIR}")
        return
    print(f"Found {len(pairs)} NOR_data.csv files")

    frames: List[pd.DataFrame] = []
    total_files = 0

    # Load each CSV and annotate with its kage ID and source file path.
    for kage_id, csv_path in pairs:
        df = read_csv_with_fallbacks(csv_path)
        if df is None or df.empty:
            # Skip when unreadable or legitimately empty.
            print(f"Skipping empty/unreadable: {csv_path}")
            continue
        total_files += 1
        # Prepend metadata columns so they're visible first in the output.
        df.insert(0, "KageID", kage_id)              # e.g., 'Kage3'
        df.insert(1, "SourceFile", str(csv_path))    # Full absolute path to the source CSV
        frames.append(df)

    # Ensure we have data to combine.
    if not frames:
        print("No readable CSV data found after loading files.")
        return

    # Concatenate all loaded frames into a single DataFrame.
    combined = pd.concat(frames, ignore_index=True, sort=False)

    # Reorder columns to keep 'KageID' and 'SourceFile' at the front,
    # followed by the original data columns in their existing order.
    cols = list(combined.columns)
    for col in ("KageID", "SourceFile"):
        if col in cols:
            cols.remove(col)
    combined = combined[["KageID", "SourceFile"] + cols]

    # Make sure the destination folder exists before writing the CSV.
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    print(f"Saving to: {OUTPUT_CSV}")
    try:
        # Persist the combined dataset to disk without row indices.
        combined.to_csv(OUTPUT_CSV, index=False)
        print(f"Done. Combined {total_files} files. Total rows: {len(combined):,}")
    except Exception as e:
        # Report any file I/O issues encountered during save.
        print(f"Failed to save CSV to {OUTPUT_CSV}\n{e}")

# Script entry point: runs consolidation only when executed directly.
print(f"Python: {sys.executable}")
main()

Python: c:\Users\Loukia\.conda\envs\Smart-Kages_Jiatong\python.exe
Found 16 NOR_data.csv files
Saving to: I:\SmartKages_Nov2025_onwards\2026-01-Jan-Feb_RELN-COLBOS_Coh2\DB\2026-01-Jan-Feb_RELN-COLBOS_Coh2_NOR_timestamps_AllKages_Combined.csv
Done. Combined 16 files. Total rows: 80


## Load the combined wheel-change table

The combined table has one row per wheel change.

Important columns:
- `KageID`: mouse/cage ID
- `rotation`: actual time the wheel changed
- `left_object` and `right_object`: object identity shown after the wheel changed

Object represents:
- `1` = training / sample
- `2` = test
- `4` = baseline

Because the sequence is `1, 4, 2, 4`, the two baseline periods need to be separated:
- first `4` = ITI baseline
- second `4` = post-trial baseline

For the main NOR analysis, we keep sample, ITI baseline and test, and exclude the post-trial baseline.

In [32]:
events = pd.read_csv(OUTPUT_CSV)

events["rotation"] = pd.to_datetime(events["rotation"])
events["set_time"] = pd.to_datetime(events["set_time"])

events.head()

,KageID,SourceFile,index,set_time,left_object,right_object,rotation
0,Kage1,I:\SmartKages_Nov2025_onwards\2026-01-Jan-Feb_...,0,2026-02-13 11:00:00,1,1,2026-02-13 11:07:23
1,Kage1,I:\SmartKages_Nov2025_onwards\2026-01-Jan-Feb_...,1,2026-02-14 11:00:00,4,4,2026-02-14 11:32:43
2,Kage1,I:\SmartKages_Nov2025_onwards\2026-01-Jan-Feb_...,2,2026-02-15 11:00:00,2,2,2026-02-15 11:31:40
3,Kage1,I:\SmartKages_Nov2025_onwards\2026-01-Jan-Feb_...,3,2026-02-16 12:00:00,3,3,2026-02-16 12:05:48
4,Kage1,I:\SmartKages_Nov2025_onwards\2026-01-Jan-Feb_...,4,2026-02-17 12:00:00,4,4,2026-02-17 12:16:10


In [34]:
# Label the phase of each wheel change
# Object codes:
# 1 = sample / training
# 2 = test
# 3 = object in place (OiP)
# 4 = baseline

# Since the sequence is 1, 4, 2, 3, 4, the first baseline is the ITI baseline
# and the second baseline is the post-trial baseline.

events = events.sort_values(["KageID", "rotation"]).copy()

events["object_code"] = events["left_object"]

# Count baseline occurrences within each Kage
# first 4 = ITI baseline, second 4 = post-trial baseline
events["baseline_number"] = (
    events["object_code"].eq(4)
    .groupby(events["KageID"])
    .cumsum()
)

events["Phase"] = np.select(
    [
        events["object_code"].eq(1),
        events["object_code"].eq(2),
        events["object_code"].eq(3),
        events["object_code"].eq(4) & events["baseline_number"].eq(1),
        events["object_code"].eq(4) & events["baseline_number"].ge(2),
    ],
    [
        "sample",
        "test",
        "OiP",
        "ITI_baseline",
        "post_trial_baseline",
    ],
    default="unknown"
)

events[["KageID", "set_time", "rotation", "left_object", "right_object", "baseline_number", "Phase"]].head(20)


,KageID,set_time,rotation,left_object,right_object,baseline_number,Phase
0,Kage1,2026-02-13 11:00:00,2026-02-13 11:07:23,1,1,0,sample
1,Kage1,2026-02-14 11:00:00,2026-02-14 11:32:43,4,4,1,ITI_baseline
2,Kage1,2026-02-15 11:00:00,2026-02-15 11:31:40,2,2,1,test
3,Kage1,2026-02-16 12:00:00,2026-02-16 12:05:48,3,3,1,OiP
4,Kage1,2026-02-17 12:00:00,2026-02-17 12:16:10,4,4,2,post_trial_baseline
5,Kage10,2026-02-13 11:00:00,2026-02-13 11:09:36,1,1,0,sample
6,Kage10,2026-02-14 11:00:00,2026-02-14 11:00:28,4,4,1,ITI_baseline
7,Kage10,2026-02-15 11:00:00,2026-02-15 11:54:48,2,2,1,test
8,Kage10,2026-02-16 12:00:00,2026-02-16 12:03:54,3,3,1,OiP
9,Kage10,2026-02-17 12:00:00,2026-02-17 12:36:34,4,4,2,post_trial_baseline


## Convert rotation time to seconds from midnight

The `drum_output` files use timestamps in seconds from midnight.

Example:
- 11:00:00 = 39600 seconds
- 11:30:00 = 41400 seconds

So we convert the `rotation` datetime into `rotation_seconds`.

In [35]:
events["date"] = events["rotation"].dt.strftime("%Y%m%d").astype(int)

events["rotation_seconds"] = (
    events["rotation"].dt.hour * 3600
    + events["rotation"].dt.minute * 60
    + events["rotation"].dt.second
)

events[["KageID", "date", "rotation", "rotation_seconds", "Phase"]].head()

,KageID,date,rotation,rotation_seconds,Phase
0,Kage1,20260213,2026-02-13 11:07:23,40043,sample
1,Kage1,20260214,2026-02-14 11:32:43,41563,ITI_baseline
2,Kage1,20260215,2026-02-15 11:31:40,41500,test
3,Kage1,20260216,2026-02-16 12:05:48,43548,OiP
4,Kage1,20260217,2026-02-17 12:16:10,44170,post_trial_baseline


## Load kage_descriptions file

`kage_descriptions.csv` tells us:
- genotype
- whether the mouse should be included
- which side is novel during the test phase

`NOR_side` is only used for the test phase.

In [36]:
DESC_CSV = ROOT_DIR / "kage_descriptions.csv" # change the directory and name of the file accordingly

desc = pd.read_csv(DESC_CSV)
desc.columns = desc.columns.str.strip()

desc["KageID"] = desc["Kage"].str.replace("kage", "Kage", regex=False)

desc.head()

,Kage,Group,Description,NOR_side,NOR_good,KageID
0,kage1,2,RELN-Colbos,R,yes,Kage1
1,kage2,2,RELN-Colbos,R,yes,Kage2
2,kage3,1,WT,R,yes,Kage3
3,kage4,2,RELN-Colbos,L,yes,Kage4
4,kage5,1,WT,L,yes,Kage5


In [37]:
events = events.merge(
    desc[["KageID", "Description", "NOR_side", "NOR_good"]],
    on="KageID",
    how="left"
)

events = events[events["NOR_good"] == "yes"].copy()

events[["KageID", "Phase", "Description", "NOR_side", "NOR_good"]].head()

,KageID,Phase,Description,NOR_side,NOR_good
0,Kage1,sample,RELN-Colbos,R,yes
1,Kage1,ITI_baseline,RELN-Colbos,R,yes
2,Kage1,test,RELN-Colbos,R,yes
3,Kage1,OiP,RELN-Colbos,R,yes
4,Kage1,post_trial_baseline,RELN-Colbos,R,yes


## Define analysis windows between wheel changes

The Smart-Kage phase starts when the wheel rotates.

Instead of using a fixed 24-hour window, the analysis window is now:

`current wheel change → next wheel change`

This accounts for small timing differences caused by the mouse triggering the wheel change when it goes to drink.

The post-trial baseline is kept in the `events` table only so it can define the end of the test phase. It is then excluded from the main analysis table.


In [38]:
# Sort wheel changes chronologically for each mouse
events = events.sort_values(["KageID", "rotation"]).copy()

# Get the next wheel rotation for each mouse
events["next_rotation"] = (
    events.groupby("KageID")["rotation"]
    .shift(-1)
)

# Use next wheel change as the end of the analysis window
events["window_end"] = events["next_rotation"]

# For the last wheel change, fall back to 24 h
events["window_end"] = events["window_end"].fillna(
    events["rotation"] + pd.Timedelta(hours=24)
)

# For the main analysis, drop the post-trial baseline.
# It is not part of the main NOR comparison.
events_for_analysis = events[
    events["Phase"] != "post_trial_baseline"
].copy()

# Save these so the analysis/plotting notebook can reload them later
OUTPUT_DIR = ROOT_DIR / "NOR_processed"
OUTPUT_DIR.mkdir(exist_ok=True)

events.to_csv(OUTPUT_DIR / "NOR_events_with_phase_all.csv", index=False)
events_for_analysis.to_csv(OUTPUT_DIR / "NOR_events_with_phase.csv", index=False)

events[[
    "KageID",
    "Phase",
    "rotation",
    "next_rotation",
    "window_end"
]].head(20)


,KageID,Phase,rotation,next_rotation,window_end
0,Kage1,sample,2026-02-13 11:07:23,2026-02-14 11:32:43,2026-02-14 11:32:43
1,Kage1,ITI_baseline,2026-02-14 11:32:43,2026-02-15 11:31:40,2026-02-15 11:31:40
2,Kage1,test,2026-02-15 11:31:40,2026-02-16 12:05:48,2026-02-16 12:05:48
3,Kage1,OiP,2026-02-16 12:05:48,2026-02-17 12:16:10,2026-02-17 12:16:10
4,Kage1,post_trial_baseline,2026-02-17 12:16:10,NaT,2026-02-18 12:16:10
5,Kage10,sample,2026-02-13 11:09:36,2026-02-14 11:00:28,2026-02-14 11:00:28
6,Kage10,ITI_baseline,2026-02-14 11:00:28,2026-02-15 11:54:48,2026-02-15 11:54:48
7,Kage10,test,2026-02-15 11:54:48,2026-02-16 12:03:54,2026-02-16 12:03:54
8,Kage10,OiP,2026-02-16 12:03:54,2026-02-17 12:36:34,2026-02-17 12:36:34
9,Kage10,post_trial_baseline,2026-02-17 12:36:34,NaT,2026-02-18 12:36:34


## Load drum_output files

The wheel-change table tells us:

- when a phase starts
- which phase it is

The drum output files tell us:

- when exploration occurred
- whether the mouse explored the left or right side

The goal is to extract exploration occurring after each wheel change.

In [39]:
drum_files = [
    f for f in sorted(ROOT_DIR.glob("kage*/analysis/drum_output/*.csv"))
    if not f.name.startswith("._")
]

print(f"Found {len(drum_files)} drum files")

Found 16 drum files


## Create a lookup table for drum files

This allows us to quickly find the correct drum file for each Kage.

In [40]:
drum_map = {
    f.parent.parent.parent.name.replace("kage", "Kage"): f
    for f in drum_files
}

list(drum_map.items())[:5]

[('Kage1',
  WindowsPath('I:/SmartKages_Nov2025_onwards/2026-01-Jan-Feb_RELN-COLBOS_Coh2/DB/kage1/analysis/drum_output/kage1_drum.csv')),
 ('Kage10',
  WindowsPath('I:/SmartKages_Nov2025_onwards/2026-01-Jan-Feb_RELN-COLBOS_Coh2/DB/kage10/analysis/drum_output/kage10_drum.csv')),
 ('Kage11',
  WindowsPath('I:/SmartKages_Nov2025_onwards/2026-01-Jan-Feb_RELN-COLBOS_Coh2/DB/kage11/analysis/drum_output/kage11_drum.csv')),
 ('Kage12',
  WindowsPath('I:/SmartKages_Nov2025_onwards/2026-01-Jan-Feb_RELN-COLBOS_Coh2/DB/kage12/analysis/drum_output/kage12_drum.csv')),
 ('Kage13',
  WindowsPath('I:/SmartKages_Nov2025_onwards/2026-01-Jan-Feb_RELN-COLBOS_Coh2/DB/kage13/analysis/drum_output/kage13_drum.csv'))]

# Extract exploration between wheel changes

For each wheel change, this extracts object exploration events from the matching drum file until the next wheel change.

This is used instead of a fixed 24-hour window because wheel-change timing can vary slightly depending on when the mouse goes to drink.

For the main analysis, the post-trial baseline is excluded, so the phases kept are:
- sample
- ITI_baseline
- test

In [41]:
# Test one wheel change first

row = events_for_analysis.iloc[0]

kage = row["KageID"]

print(kage)
print(row["Phase"])
print("Start:", row["rotation"])
print("End:", row["window_end"])

drum = pd.read_csv(drum_map[kage])

drum["datetime"] = (
    pd.to_datetime(drum["date"].astype(str), format="%Y%m%d")
    + pd.to_timedelta(drum["timestamp"], unit="s")
)

start = row["rotation"]
end = row["window_end"]

session = drum[
    (drum["datetime"] >= start) &
    (drum["datetime"] < end)
].copy()

print(len(session))
session.head()

Kage1
sample
Start: 2026-02-13 11:07:23
End: 2026-02-14 11:32:43
1321


,date,timestamp,frame,side,climbing,datetime
31768,20260213,40848.5,2363,left,False,2026-02-13 11:20:48.500
31769,20260213,40849.0,2364,left,False,2026-02-13 11:20:49.000
31770,20260213,40849.5,2365,left,False,2026-02-13 11:20:49.500
31771,20260213,40850.0,2366,left,False,2026-02-13 11:20:50.000
31772,20260213,40850.5,2367,left,False,2026-02-13 11:20:50.500


## Build the binned exploration table between wheel changes

For every included wheel change, this loop:
1. Loads the correct drum file.
2. Extracts exploration after the current wheel rotation and before the next wheel rotation.
3. Calculates time from wheel change.
4. Bins exploration into time windows.
5. Counts left and right exploration.
6. Converts counts into seconds.

The post-trial baseline is excluded from this table.

In [42]:
BIN_SIZE = 60          # 1-minute bins
ROW_DURATION = 0.5     # each row = 0.5 sec

all_binned = []

for _, row in events_for_analysis.iterrows():

    kage = row["KageID"]

    if kage not in drum_map:
        print("Missing drum file:", kage)
        continue

    drum = pd.read_csv(drum_map[kage])

    drum["datetime"] = (
        pd.to_datetime(drum["date"].astype(str), format="%Y%m%d")
        + pd.to_timedelta(drum["timestamp"], unit="s")
    )

    start = row["rotation"]
    end = row["window_end"]

    session = drum[
        (drum["datetime"] >= start) &
        (drum["datetime"] < end)
    ].copy()

    if session.empty:
        print("No exploration:", kage, row["Phase"], start)
        continue

    session["time_from_rotation_sec"] = (
        session["datetime"] - start
    ).dt.total_seconds()

    session["bin"] = (
        session["time_from_rotation_sec"] // BIN_SIZE
    ).astype(int)

    binned = pd.crosstab(session["bin"], session["side"])

    for side in ["left", "right"]:
        if side not in binned.columns:
            binned[side] = 0

    binned = binned.reset_index()

    binned["left_sec"] = binned["left"] * ROW_DURATION
    binned["right_sec"] = binned["right"] * ROW_DURATION
    binned["total_sec"] = binned["left_sec"] + binned["right_sec"]

    binned["KageID"] = kage
    binned["Genotype"] = row["Description"]
    binned["Phase"] = row["Phase"]
    binned["rotation"] = start
    binned["window_end"] = end
    binned["NOR_side"] = row["NOR_side"]

    all_binned.append(binned)

binned_df = pd.concat(all_binned, ignore_index=True)

binned_df.head()


side,bin,left,right,left_sec,right_sec,total_sec,KageID,Genotype,Phase,rotation,window_end,NOR_side
0,13,61,0,30.5,0.0,30.5,Kage1,RELN-Colbos,sample,2026-02-13 11:07:23,2026-02-14 11:32:43,R
1,14,19,0,9.5,0.0,9.5,Kage1,RELN-Colbos,sample,2026-02-13 11:07:23,2026-02-14 11:32:43,R
2,15,28,0,14.0,0.0,14.0,Kage1,RELN-Colbos,sample,2026-02-13 11:07:23,2026-02-14 11:32:43,R
3,16,80,24,40.0,12.0,52.0,Kage1,RELN-Colbos,sample,2026-02-13 11:07:23,2026-02-14 11:32:43,R
4,17,0,24,0.0,12.0,12.0,Kage1,RELN-Colbos,sample,2026-02-13 11:07:23,2026-02-14 11:32:43,R


In [43]:
OUTPUT_DIR = ROOT_DIR / "NOR_processed"
OUTPUT_DIR.mkdir(exist_ok=True)

# Main corrected output: sample, ITI_baseline and test only
binned_df.to_csv(OUTPUT_DIR / "NOR_binned_between_wheel_changes.csv", index=False)

# Also save using the old filename so the analysis notebook still works
binned_df.to_csv(OUTPUT_DIR / "NOR_24h_binned_by_phase.csv", index=False)

## Total exploration by phase 

This collapses the bins into one total exploration value per mouse per phase.

In [44]:
phase_total = (
    binned_df
    .groupby(["KageID", "Genotype", "Phase"], as_index=False)["total_sec"]
    .sum()
)

phase_total.head()

,KageID,Genotype,Phase,total_sec
0,Kage1,RELN-Colbos,ITI_baseline,431.5
1,Kage1,RELN-Colbos,OiP,440.5
2,Kage1,RELN-Colbos,sample,660.5
3,Kage1,RELN-Colbos,test,431.0
4,Kage10,RELN-Colbos,ITI_baseline,644.5


In [45]:
phase_total.to_csv(OUTPUT_DIR / "NOR_total_exploration_by_phase.csv", index=False)
phase_total.to_csv(OUTPUT_DIR / "NOR_24h_total_exploration_by_phase.csv", index=False)

## Cumulative exploration over time

This shows how total exploration builds up after each wheel change.

For each mouse and phase, we cumulatively sum exploration across time bins.

In [46]:
binned_df = binned_df.sort_values(["KageID", "Phase", "rotation", "bin"]).copy()

binned_df["cumulative_left_sec"] = (
    binned_df.groupby(["KageID", "Phase", "rotation"])["left_sec"].cumsum()
)

binned_df["cumulative_right_sec"] = (
    binned_df.groupby(["KageID", "Phase", "rotation"])["right_sec"].cumsum()
)

binned_df["cumulative_total_sec"] = (
    binned_df.groupby(["KageID", "Phase", "rotation"])["total_sec"].cumsum()
)

binned_df.head()

side,bin,left,right,left_sec,right_sec,total_sec,KageID,Genotype,Phase,rotation,window_end,NOR_side,cumulative_left_sec,cumulative_right_sec,cumulative_total_sec
97,0,0,1,0.0,0.5,0.5,Kage1,RELN-Colbos,ITI_baseline,2026-02-14 11:32:43,2026-02-15 11:31:40,R,0.0,0.5,0.5
98,1,3,60,1.5,30.0,31.5,Kage1,RELN-Colbos,ITI_baseline,2026-02-14 11:32:43,2026-02-15 11:31:40,R,1.5,30.5,32.0
99,2,33,0,16.5,0.0,16.5,Kage1,RELN-Colbos,ITI_baseline,2026-02-14 11:32:43,2026-02-15 11:31:40,R,18.0,30.5,48.5
100,3,18,20,9.0,10.0,19.0,Kage1,RELN-Colbos,ITI_baseline,2026-02-14 11:32:43,2026-02-15 11:31:40,R,27.0,40.5,67.5
101,4,0,6,0.0,3.0,3.0,Kage1,RELN-Colbos,ITI_baseline,2026-02-14 11:32:43,2026-02-15 11:31:40,R,27.0,43.5,70.5


In [47]:
binned_df.to_csv(OUTPUT_DIR / "NOR_binned_between_wheel_changes_with_cumulative.csv", index=False)

# Also save using the old filename so the analysis notebook still works
binned_df.to_csv(OUTPUT_DIR / "NOR_24h_binned_by_phase_with_cumulative.csv", index=False)

## Convert left/right exploration into novel/familiar exploration

The drum file records left and right exploration.

For test sessions we convert these into exploration of the novel object and familiar object using the NOR_side information from kage_descriptions.

In [48]:
test_df = binned_df[
    binned_df["Phase"] == "test"
].copy()

In [49]:
test_df["NOR_side"].value_counts()

NOR_side
R    746
L    716
Name: count, dtype: int64

In [50]:
test_df["novel_sec"] = np.where(
    test_df["NOR_side"] == "L",
    test_df["left_sec"],
    test_df["right_sec"]
)

test_df["familiar_sec"] = np.where(
    test_df["NOR_side"] == "L",
    test_df["right_sec"],
    test_df["left_sec"]
)

## Discrimination Index (DI)

DI measures preference for the novel object.

DI = (Novel - Familiar) / (Novel + Familiar)

- DI > 0 = novel preference
- DI = 0 = no preference
- DI < 0 = familiar preference

In [51]:
test_df["DI"] = (
    test_df["novel_sec"] - test_df["familiar_sec"]
) / (
    test_df["novel_sec"] + test_df["familiar_sec"]
)

test_df["DI"] = test_df["DI"].replace(
    [np.inf, -np.inf],
    np.nan
)

di_table = test_df[
    [
        "KageID",
        "Genotype",
        "Phase",
        "rotation",
        "bin",
        "novel_sec",
        "familiar_sec",
        "DI"
    ]
].copy()

di_table.head()

di_table.to_csv(
    OUTPUT_DIR / "NOR_test_DI_per_mouse.csv",
    index=False
)


## Test only 2-hour analysis

Classical NOR analyses focus on behaviour shortly after object presentation.

This dataframe contains only the first 2 hours after wheel rotation during the test phase.

In [52]:
test_2h = test_df[
    test_df["bin"] < 120
].copy()

# because per-bin DI already exists, we need to calculate a single DI score per mouse
test_2h_mouse_DI = (
    test_2h
    .groupby(
        ["KageID","Genotype"],
        as_index=False
    )
    .agg(
        novel_sec=("novel_sec","sum"),
        familiar_sec=("familiar_sec","sum")
    )
)

# calculate DI
test_2h_mouse_DI["DI"] = (
    test_2h_mouse_DI["novel_sec"]
    -
    test_2h_mouse_DI["familiar_sec"]
) / (
    test_2h_mouse_DI["novel_sec"]
    +
    test_2h_mouse_DI["familiar_sec"]
)

test_2h_mouse_DI.head()

# for every 1-minute bin in the first 2 hours
test_2h.to_csv(
    OUTPUT_DIR / "NOR_test_first2hours_binned.csv",
    index=False
)

# mouse-level DI summary
test_2h_mouse_DI.to_csv(
    OUTPUT_DIR / "NOR_test_first2hours_mouse_DI.csv",
    index=False
)


## First approach analysis

For each mouse, we can identify the first exploration event occurring after the test-phase wheel rotation. 
The side of this first exploration event (left or right) was compared with the known location of the novel object obtained from the kage_descriptions metadata.

This analysis provides a simple behavioural measure of novelty preference that is independent of total exploration duration or discrimination index calculations.

In [53]:
first_approach_results = []

for _, row in events.iterrows():

    if row["Phase"] != "test":
        continue

    kage = row["KageID"]

    if kage not in drum_map:
        continue

    drum = pd.read_csv(drum_map[kage])

    drum["datetime"] = (
        pd.to_datetime(
            drum["date"].astype(str),
            format="%Y%m%d"
        )
        +
        pd.to_timedelta(
            drum["timestamp"],
            unit="s"
        )
    )

    start = row["rotation"]

    end = start + pd.Timedelta(hours=2)

    session = drum[
        (drum["datetime"] >= start)
        &
        (drum["datetime"] < end)
    ].copy()

    if len(session) == 0:
        continue

    session = session.sort_values("datetime")

    first_side = session.iloc[0]["side"]

    if row["NOR_side"] == "L":
        first_approach = (
            "novel"
            if first_side == "left"
            else "familiar"
        )
    else:
        first_approach = (
            "novel"
            if first_side == "right"
            else "familiar"
        )

    first_approach_results.append(
        {
            "KageID": kage,
            "Genotype": row["Description"],
            "NOR_side": row["NOR_side"],
            "FirstSide": first_side,
            "FirstApproach": first_approach
        }
    )

In [54]:
first_approach_df = pd.DataFrame(
    first_approach_results
)

first_approach_df.head()

,KageID,Genotype,NOR_side,FirstSide,FirstApproach
0,Kage1,RELN-Colbos,R,right,novel
1,Kage10,RELN-Colbos,R,right,novel
2,Kage11,WT,L,left,novel
3,Kage12,WT,L,left,novel
4,Kage13,WT,R,left,familiar


In [55]:
first_approach_df.to_csv(
    OUTPUT_DIR / "NOR_test_first_approach.csv",
    index=False
)

## Build master tables across all cohorts

The cells above process one cohort at a time.

This section combines the processed outputs from all cohort folders into master CSV files.

This is needed because `Kage1`, `Kage2`, etc. repeat across cohorts, so each row gets:

- `Cohort`: cohort folder name
- `Age`: inferred age group (`6m`, `12m`, `24m`)
- `MouseSessionID`: unique mouse/session ID made from cohort + KageID

Before running this section, each cohort should already have its own `NOR_processed` folder created by running the postprocessing workflow above.

In [12]:
# Master cohort settings

BASE_DIR = Path("I:/SmartKages_Nov2025_onwards")

MASTER_OUTPUT = BASE_DIR / "NOR_MASTER"
MASTER_OUTPUT.mkdir(exist_ok=True)

# Automatically find cohort folders that already have NOR_processed outputs
COHORT_DIRS = sorted([
    cohort_dir / "DB"
    for cohort_dir in BASE_DIR.iterdir()
    if cohort_dir.is_dir()
    and (cohort_dir / "DB" / "NOR_processed").exists()
])

print(f"Found {len(COHORT_DIRS)} processed cohort folders:")
for cohort_db in COHORT_DIRS:
    print("-", cohort_db.parent.name)

Found 1 processed cohort folders:
- 2026-01-Jan-Feb_RELN-COLBOS_Coh2


In [13]:
def infer_age_from_cohort(cohort_name: str) -> str:
    return "3m"


def load_cohort_output(cohort_db: Path, filename: str) -> pd.DataFrame | None:
    """
    Load one processed output file from one cohort.
    Adds Cohort, Age and MouseSessionID.
    Returns None if the file does not exist.
    """
    cohort_name = cohort_db.parent.name
    age = infer_age_from_cohort(cohort_name)
    path = cohort_db / "NOR_processed" / filename

    if not path.exists():
        print(f"Missing {filename} for {cohort_name}")
        return None

    df = pd.read_csv(path)

    df["Cohort"] = cohort_name
    df["Age"] = age

    if "KageID" in df.columns:
        df["MouseSessionID"] = (
            df["Cohort"].astype(str)
            + "_"
            + df["KageID"].astype(str)
        )

    return df

In [14]:
# Files to combine across cohorts

FILES_TO_COMBINE = {
    "binned": "NOR_24h_binned_by_phase_with_cumulative.csv",
    "phase_total": "NOR_24h_total_exploration_by_phase.csv",
    "test_di_per_mouse": "NOR_test_DI_per_mouse.csv",
    "test_first2h_binned": "NOR_test_first2hours_binned.csv",
    "test_first2h_mouse_DI": "NOR_test_first2hours_mouse_DI.csv",
    "first_approach": "NOR_test_first_approach.csv",
    "events": "NOR_events_with_phase.csv",
    "events_all": "NOR_events_with_phase_all.csv",
}

master_tables = {}

for table_name, filename in FILES_TO_COMBINE.items():

    frames = []

    for cohort_db in COHORT_DIRS:
        df = load_cohort_output(cohort_db, filename)
        if df is not None and not df.empty:
            frames.append(df)

    if frames:
        master_tables[table_name] = pd.concat(frames, ignore_index=True)
        print(table_name, master_tables[table_name].shape)
    else:
        print(f"No data combined for {table_name}")

binned (5860, 18)
phase_total (64, 7)
test_di_per_mouse (1462, 11)
test_first2h_binned (304, 21)
test_first2h_mouse_DI (16, 8)
first_approach (16, 8)
events (64, 20)
events_all (80, 20)


In [15]:
# Save master tables

for table_name, df in master_tables.items():
    out_path = MASTER_OUTPUT / f"NOR_MASTER_{table_name}.csv"
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)

# Quick check of the main master table
if "binned" in master_tables:
    display(master_tables["binned"].head())
    display(master_tables["binned"].groupby(["Age", "Cohort", "Genotype", "Phase"]).size().reset_index(name="n_rows").head(20))

Saved: I:\SmartKages_Nov2025_onwards\NOR_MASTER\NOR_MASTER_binned.csv
Saved: I:\SmartKages_Nov2025_onwards\NOR_MASTER\NOR_MASTER_phase_total.csv
Saved: I:\SmartKages_Nov2025_onwards\NOR_MASTER\NOR_MASTER_test_di_per_mouse.csv
Saved: I:\SmartKages_Nov2025_onwards\NOR_MASTER\NOR_MASTER_test_first2h_binned.csv
Saved: I:\SmartKages_Nov2025_onwards\NOR_MASTER\NOR_MASTER_test_first2h_mouse_DI.csv
Saved: I:\SmartKages_Nov2025_onwards\NOR_MASTER\NOR_MASTER_first_approach.csv
Saved: I:\SmartKages_Nov2025_onwards\NOR_MASTER\NOR_MASTER_events.csv
Saved: I:\SmartKages_Nov2025_onwards\NOR_MASTER\NOR_MASTER_events_all.csv


,bin,left,right,left_sec,right_sec,total_sec,KageID,Genotype,Phase,rotation,window_end,NOR_side,cumulative_left_sec,cumulative_right_sec,cumulative_total_sec,Cohort,Age,MouseSessionID
0,0,0,1,0.0,0.5,0.5,Kage1,RELN-Colbos,ITI_baseline,2026-02-14 11:32:43,2026-02-15 11:31:40,R,0.0,0.5,0.5,2026-01-Jan-Feb_RELN-COLBOS_Coh2,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2_Kage1
1,1,3,60,1.5,30.0,31.5,Kage1,RELN-Colbos,ITI_baseline,2026-02-14 11:32:43,2026-02-15 11:31:40,R,1.5,30.5,32.0,2026-01-Jan-Feb_RELN-COLBOS_Coh2,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2_Kage1
2,2,33,0,16.5,0.0,16.5,Kage1,RELN-Colbos,ITI_baseline,2026-02-14 11:32:43,2026-02-15 11:31:40,R,18.0,30.5,48.5,2026-01-Jan-Feb_RELN-COLBOS_Coh2,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2_Kage1
3,3,18,20,9.0,10.0,19.0,Kage1,RELN-Colbos,ITI_baseline,2026-02-14 11:32:43,2026-02-15 11:31:40,R,27.0,40.5,67.5,2026-01-Jan-Feb_RELN-COLBOS_Coh2,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2_Kage1
4,4,0,6,0.0,3.0,3.0,Kage1,RELN-Colbos,ITI_baseline,2026-02-14 11:32:43,2026-02-15 11:31:40,R,27.0,43.5,70.5,2026-01-Jan-Feb_RELN-COLBOS_Coh2,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2_Kage1


,Age,Cohort,Genotype,Phase,n_rows
0,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2,RELN-Colbos,ITI_baseline,557
1,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2,RELN-Colbos,OiP,663
2,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2,RELN-Colbos,sample,683
3,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2,RELN-Colbos,test,699
4,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2,WT,ITI_baseline,674
5,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2,WT,OiP,877
6,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2,WT,sample,944
7,3m,2026-01-Jan-Feb_RELN-COLBOS_Coh2,WT,test,763


In [175]:
# Optional: create master total exploration again from the combined binned table
# This makes sure the summary uses the same columns across all cohorts.

if "binned" in master_tables:
    master_phase_total = (
        master_tables["binned"]
        .groupby(["Age", "Cohort", "MouseSessionID", "KageID", "Genotype", "Phase"], as_index=False)["total_sec"]
        .sum()
    )

    master_phase_total.to_csv(
        MASTER_OUTPUT / "NOR_MASTER_total_exploration_by_phase_recomputed.csv",
        index=False
    )

    display(master_phase_total.head())

,Age,Cohort,MouseSessionID,KageID,Genotype,Phase,total_sec
0,12m,2024-07-jul-sep_NLGF12m,2024-07-jul-sep_NLGF12m_Kage1,Kage1,WT,ITI_baseline,1192.5
1,12m,2024-07-jul-sep_NLGF12m,2024-07-jul-sep_NLGF12m_Kage1,Kage1,WT,sample,1378.0
2,12m,2024-07-jul-sep_NLGF12m,2024-07-jul-sep_NLGF12m_Kage1,Kage1,WT,test,1235.0
3,12m,2024-07-jul-sep_NLGF12m,2024-07-jul-sep_NLGF12m_Kage10,Kage10,APP1-em1BDS,ITI_baseline,138.5
4,12m,2024-07-jul-sep_NLGF12m,2024-07-jul-sep_NLGF12m_Kage10,Kage10,APP1-em1BDS,sample,279.0
